# 📈 Prediksi Time-Series (Auto-ARIMA)

Ramalkan nilai masa depan (penjualan, suhu, harga, jumlah pengunjung…) dengan
**pmdarima `auto_arima`** — mencari model ARIMA terbaik **otomatis**, tanpa
tuning manual. Data contoh dibuat di sel (ganti dengan CSV-mu).

**Yang kamu pelajari:** dekomposisi pola, pemilihan model otomatis, forecast
dengan interval kepercayaan, dan evaluasi MAE/MAPE.

In [ ]:
# Sel 1 — data contoh: penjualan harian 2 tahun (tren + musiman mingguan).
# Ganti dengan datamu:  df = pd.read_csv("data.csv", parse_dates=["tanggal"])
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
hari = pd.date_range("2024-07-01", periods=730, freq="D")
tren = np.linspace(100, 180, 730)
musiman = 25 * np.sin(2 * np.pi * np.arange(730) / 7)        # pola mingguan
noise = rng.normal(0, 8, 730)
df = pd.DataFrame({"tanggal": hari, "penjualan": tren + musiman + noise})
df = df.set_index("tanggal")
df.plot(figsize=(11, 3), title="Penjualan harian (2 tahun)");

In [ ]:
# Sel 2 — bagi data latih/uji & cari model ARIMA terbaik OTOMATIS.
from pmdarima import auto_arima

latih, uji = df.iloc[:-30], df.iloc[-30:]     # 30 hari terakhir utk evaluasi
model = auto_arima(
    latih["penjualan"], seasonal=True, m=7,   # musiman mingguan
    stepwise=True, suppress_warnings=True, trace=True,
)
print("\nModel terpilih:", model.order, "musiman:", model.seasonal_order)

In [ ]:
# Sel 3 — forecast 30 hari + interval kepercayaan 95%, lalu evaluasi.
import matplotlib.pyplot as plt

ramal, ik = model.predict(n_periods=30, return_conf_int=True)
ramal = pd.Series(ramal, index=uji.index)

mae = (ramal - uji["penjualan"]).abs().mean()
mape = ((ramal - uji["penjualan"]).abs() / uji["penjualan"]).mean() * 100
print(f"MAE = {mae:.1f}   MAPE = {mape:.1f}%")

fig, ax = plt.subplots(figsize=(11, 4))
latih["penjualan"].iloc[-120:].plot(ax=ax, label="data latih")
uji["penjualan"].plot(ax=ax, label="aktual", color="black")
ramal.plot(ax=ax, label="forecast", color="crimson")
ax.fill_between(uji.index, ik[:, 0], ik[:, 1], alpha=0.2, color="crimson", label="IK 95%")
ax.legend(); ax.set_title("Forecast 30 hari ke depan"); plt.tight_layout()

**Langkah lanjut:** coba `prophet` (juga terpasang) untuk data dengan hari
libur, atau `sktime` untuk pipeline ML time-series penuh (klasifikasi, regresi,
validasi silang temporal).